# Snowflake Crime Data ETL Pipeline Project

## Introduction

This project was developed to support operational crime monitoring and reporting across UK police forces. The objective of this project was to design and implement a scalable Python-based, Snowflake data ETL pipeline that transforms raw crime data into a clean, aggregated, and reporting-ready dataset suitable for use in BI analysis. 

The pipleline processes UK Police crime data for four selected police forces and integrates multiple enrichment datasets. These enrichment datasets allow for more meaningful comparisons between police forces across demographic and socioeconomic factors. 

To support scalable and reliable reporting, the pipeline was structured into distinct engineering stages including: 

• Batch ingestion of raw crime data  
• Enrichment dataset integration  
• Data cleaning and validation  
• Feature engineering and transformation  
• Aggregation to final reporting grain  
• and Export of final BI-ready reporting dataset 

Particular emphasis was placed on data quality and validation throughout the pipeline. Validation checks were implemented at every stage to ensure each stage worked as intended. Reporting-grain validation was also conducted to ensure the final dataset maintained a single, consistent reporting grain. 

The final dataset was aggregated to reporting grain of: 

***Force x Year x Quater x District x Crime Type***

This grain was chosen to support time-series analysis, force-level and district comparisons, and crime category monitoring. 

## Workflow Diagram 

*insert workflow diagram

## Data Assumptions - *replace with THIS project's specific assumptions

- assumption 1
- assumption 2

## Import Python Libraries 

In [ ]:
#data handling libraries 
import pandas as pd 
import numpy as np

#any other libraries go 
from datetime import datetime, timezone
import re
import uuid

## Ingestion Layer - edit!

**Outline:** 

This layer follows the steps taken to extract crime and police force data from data.police.uk, as well as ONS deprivation and population data for later cleaning and transformation. 

The following ingestion tasks were completed: 

- Created an AWS account and shared S3 bucket used by the group.
- Created an IAM user and permissions policy that allowed the local Python notebooks to upload files to the required locations in the bucket.
- Created a shared AWS IAM role that allows the team’s separate Snowflake accounts to access the S3 bucket.
- Organised the bucket into separate prefixes for raw crime data, cleaned crime data, population data, English deprivation data and Welsh deprivation data.
- Wrote Python notebooks containing functions that download each dataset and upload them to the appropriate S3 prefixes.
- Created and tested Snowflake storage integration, file formats, and external stages to confirm that the files could be accessed from the bucket.

*As Snowflake trial account usage is limited, the ingestion of crime and police force data from data.police.uk has been completed in Jupyter Notebook. 

Refer to the following files for the complete crime data ingestion process: 

- Crime Data Ingestion Function: 'Snowflake_Crime_Data_ETL_Pipeline/<folder/branch>/ingestion.py'
- Crime Data Ingestion: 'Snowflake_Crime_Data_ETL_Pipeline/<folder/branch>/notebook.py'

### Step 1: Create Storage Integration 

A storage integration was created to store raw & clean crime, population, and deprivation datasets. 

Purpose:  
 
- Connect Snowflake to the raw and clean S3 locations.  

Important:
- This script does not alter or remove files from the raw S3 location.
- CREATE and COPY operations only create Snowflake objects and read the source files.
- The storage integration may already exist. If it does, ALTER is used to retain the complete list of approved S3 locations.

In [ ]:
%%sql -r dataframe_1
/* Account-level objects and permissions require ACCOUNTADMIN. */
USE ROLE ACCOUNTADMIN;

CREATE STORAGE INTEGRATION IF NOT EXISTS CRIME_S3_INTEGRATION
    TYPE = EXTERNAL_STAGE
    STORAGE_PROVIDER = 'S3'
    ENABLED = TRUE
    STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::559852958324:role/SnowflakeCrimeS3Role'
    STORAGE_ALLOWED_LOCATIONS = (
        's3://rockborne-ch19-g1-crime/raw/uk-police/',
        's3://rockborne-ch19-g1-crime/clean/uk-police/',
        's3://rockborne-ch19-g1-crime/raw/enrichment/population/',
        's3://rockborne-ch19-g1-crime/raw/enrichment/deprivation/england/',
        's3://rockborne-ch19-g1-crime/raw/enrichment/deprivation/wales/'
    );


GRANT USAGE ON INTEGRATION CRIME_S3_INTEGRATION TO ROLE SYSADMIN;

### Step 2: Create Crime Database

Purpose:  
 
- Create the database objects required by the cleaning & validation layer.  
- Load the raw street-crime CSV files into CRIME_ETL_DB.RAW.STREET_CRIME.

In [ ]:
%%sql -r dataframe_2
/* Create the project database and allow SYSADMIN to create its schemas. */
CREATE DATABASE IF NOT EXISTS CRIME_ETL_DB;
GRANT USAGE ON DATABASE CRIME_ETL_DB TO ROLE SYSADMIN;
GRANT CREATE SCHEMA ON DATABASE CRIME_ETL_DB TO ROLE SYSADMIN;

/* All remaining project objects are owned or operated by SYSADMIN. */
USE ROLE SYSADMIN;
USE DATABASE CRIME_ETL_DB;

CREATE SCHEMA IF NOT EXISTS RAW;
CREATE SCHEMA IF NOT EXISTS DATA_QUALITY;
CREATE SCHEMA IF NOT EXISTS CLEAN;

In [ ]:
%%sql -r dataframe_3
USE SCHEMA CRIME_ETL_DB.RAW;

/*
    Police.uk street-crime files contain one header row.
    Blank fields are converted to NULL and quoted commas are handled correctly.
*/
CREATE FILE FORMAT IF NOT EXISTS CRIME_CSV_FORMAT
    TYPE = CSV
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    TRIM_SPACE = TRUE
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

/* External stage used to read the untouched police files in S3. */
CREATE STAGE IF NOT EXISTS CRIME_S3_STAGE
    URL = 's3://rockborne-ch19-g1-crime/raw/uk-police/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION
    FILE_FORMAT = CRIME_CSV_FORMAT;

/* Confirm that Snowflake can see the source files before loading them. */
LIST @CRIME_ETL_DB.RAW.CRIME_S3_STAGE;

In [ ]:
%%sql -r dataframe_4
/*
    Raw landing table expected by Police_Crime_Snowflake_Cleaning_Validation.ipynb.
    Text is retained for most source fields so cleaning and type validation happen
    in the cleaning layer rather than silently changing the source values on load.
*/
CREATE TABLE IF NOT EXISTS CRIME_ETL_DB.RAW.STREET_CRIME (
    "Crime ID" TEXT,
    "Month" TEXT,
    "Reported by" TEXT,
    "Falls within" TEXT,
    "Longitude" TEXT,
    "Latitude" TEXT,
    "Location" TEXT,
    "LSOA code" TEXT,
    "LSOA name" TEXT,
    "Crime type" TEXT,
    "Last outcome category" TEXT,
    "Context" TEXT
);

/*
    Load only the four police forces included in this project.
    PATTERN searches file paths and names case-insensitively.
    FORCE = FALSE avoids reloading files already recorded in Snowflake load history.
*/
COPY INTO CRIME_ETL_DB.RAW.STREET_CRIME
FROM @CRIME_ETL_DB.RAW.CRIME_S3_STAGE
FILE_FORMAT = (FORMAT_NAME = CRIME_ETL_DB.RAW.CRIME_CSV_FORMAT)
PATTERN = '.*(metropolitan|west-midlands|south-wales|sussex).*street\\.csv'
ON_ERROR = 'CONTINUE'
FORCE = FALSE;

In [ ]:
%%sql -r dataframe_5
/* Basic checks after the load. */
SELECT COUNT(*) AS RAW_ROWS
FROM CRIME_ETL_DB.RAW.STREET_CRIME;

SELECT
    "Falls within" AS FORCE_NAME,
    COUNT(*) AS RAW_ROWS
FROM CRIME_ETL_DB.RAW.STREET_CRIME
GROUP BY "Falls within"
ORDER BY FORCE_NAME;

In [ ]:
/*
    Shared clean external stage used by both crime & supplementry data cleaning steps for their CSV exports.
    Creating a stage does not write anything to S3; COPY INTO the stage performs
    the export later.
*/
USE SCHEMA CRIME_ETL_DB.CLEAN;

CREATE STAGE IF NOT EXISTS CRIME_CLEAN_STAGE
    URL = 's3://rockborne-ch19-g1-crime/clean/uk-police/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION;

LIST @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE;

### Step 3: Create Supplementary Data File Formats & Schemas

Datasets:
- Police-force-area population data
- England Index of Multiple Deprivation (IMD)
 - Wales Index of Multiple Deprivation (WIMD)

The supplementary cleaning step reads these external stages directly. It does not require separate raw Snowflake tables.

Important:
- The raw S3 files remain untouched.
- This script only creates file formats and references to the S3 locations.
- The source files must be CSV files before the notebook is run.

In [ ]:
/*
    Keep the header rows available to the notebook. Its parsing logic explicitly
    identifies and excludes headers while retaining source-row visibility.
*/
CREATE FILE FORMAT IF NOT EXISTS POPULATION_CSV_FORMAT
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    TRIM_SPACE = TRUE
    SKIP_HEADER = 0
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

/*
    Population stage expected by the notebook.
    The original workbook must be exported to CSV and placed in this /csv/ folder.
*/
CREATE STAGE IF NOT EXISTS POPULATION_S3_STAGE
    URL = 's3://rockborne-ch19-g1-crime/raw/enrichment/population/csv/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION
    FILE_FORMAT = POPULATION_CSV_FORMAT;

In [ ]:
CREATE FILE FORMAT IF NOT EXISTS DEPRIVATION_ENGLAND_CSV_FORMAT
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    TRIM_SPACE = TRUE
    SKIP_HEADER = 0
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

/* England IMD stage expected by the notebook. */
CREATE STAGE IF NOT EXISTS DEPRIVATION_ENGLAND_S3_STAGE
    URL = 's3://rockborne-ch19-g1-crime/raw/enrichment/deprivation/england/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION
    FILE_FORMAT = DEPRIVATION_ENGLAND_CSV_FORMAT;

In [ ]:
CREATE FILE FORMAT IF NOT EXISTS DEPRIVATION_WALES_CSV_FORMAT
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    EMPTY_FIELD_AS_NULL = TRUE
    TRIM_SPACE = TRUE
    SKIP_HEADER = 0
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

/* Wales WIMD stage expected by the notebook. */
CREATE STAGE IF NOT EXISTS DEPRIVATION_WALES_S3_STAGE
    URL = 's3://rockborne-ch19-g1-crime/raw/enrichment/deprivation/wales/'
    STORAGE_INTEGRATION = CRIME_S3_INTEGRATION
    FILE_FORMAT = DEPRIVATION_WALES_CSV_FORMAT;

In [ ]:
/* Confirm that Snowflake can see the files in each source location. */
LIST @CRIME_ETL_DB.RAW.POPULATION_S3_STAGE;
LIST @CRIME_ETL_DB.RAW.DEPRIVATION_ENGLAND_S3_STAGE;
LIST @CRIME_ETL_DB.RAW.DEPRIVATION_WALES_S3_STAGE;

In [ ]:
/*
    Preview the population source. The notebook expects:
    $1 = police-force-area code
    $2 = police-force-area name
    $3 = year
    $4 to $175 = female and male population counts by age.
*/
SELECT
    METADATA$FILENAME AS FILE_NAME,
    METADATA$FILE_ROW_NUMBER AS ROW_NUMBER,
    t.$1 AS PFA_CODE,
    t.$2 AS FORCE_NAME,
    t.$3 AS YEAR,
    t.$4 AS FIRST_POPULATION_FIELD,
    t.$175 AS LAST_POPULATION_FIELD
FROM @CRIME_ETL_DB.RAW.POPULATION_S3_STAGE t
LIMIT 30;

In [ ]:
/*
    Preview the England IMD source.
    The notebook uses 56 columns, beginning with the 2021 LSOA fields.
*/
SELECT
    METADATA$FILENAME AS FILE_NAME,
    METADATA$FILE_ROW_NUMBER AS ROW_NUMBER,
    t.$1 AS LSOA_CODE,
    t.$2 AS LSOA_NAME,
    t.$3 AS LOCAL_AUTHORITY_CODE,
    t.$4 AS LOCAL_AUTHORITY_NAME,
    t.$5 AS IMD_SCORE,
    t.$6 AS IMD_RANK,
    t.$7 AS IMD_DECILE,
    t.$56 AS WORKING_AGE_POPULATION
FROM @CRIME_ETL_DB.RAW.DEPRIVATION_ENGLAND_S3_STAGE t
LIMIT 30;


In [ ]:
/*
    Preview the Wales WIMD long-format source.
    The notebook uses the six analytical fields shown below and ignores the
    reference, sorting and hierarchy metadata fields.
*/
SELECT
    METADATA$FILENAME AS FILE_NAME,
    METADATA$FILE_ROW_NUMBER AS ROW_NUMBER,
    t.$1 AS DATA_VALUE,
    t.$3 AS DATA_DESCRIPTION,
    t.$7 AS AREA_CODE,
    t.$11 AS AREA_NAME,
    t.$15 AS DOMAIN,
    t.$19 AS NOTES
FROM @CRIME_ETL_DB.RAW.DEPRIVATION_WALES_S3_STAGE t
LIMIT 30;

## Cleaning & Validation Layer 

### Step 1: Police Crime Cleaning and Validation

#### Outline:

Cleaning and validation only for Metropolitan Police, West Midlands Police, South Wales Police, and Sussex Police.

This step begins with a raw Snowflake table supplied by the ingestion team and produces:

- a standardised, deduplicated clean table for downstream teammates;
- a quarantine table containing every rejected row and its rejection reason;
- run-level and force-level audit metrics;
- an explicit validation-results table.

It deliberately excludes enrichment, aggregation, trend analysis, visualisation, and Power BI export.

#### Ownership boundary

**Input contract:** one raw table containing the standard data.police.uk street-crime columns. The raw table is not changed.

**Output contract:** record-level clean and quarantine tables. A downstream teammate may enrich and aggregate the clean table.

#### Cleaning Rules

1. Trim text and turn blank strings into `NULL`.
2. Standardise the four force names and a small set of known crime-category aliases.
3. Parse `MONTH` as a first-of-month date; parse coordinates with `TRY_TO_DOUBLE`.
4. Reject rows with missing/invalid critical fields, invalid coordinates, unknown force/category, or missing location identifiers.
5. Deduplicate by `CRIME_ID`, retaining the most complete row deterministically.
6. Preserve all rejected rows in quarantine.

The original notebook dropped duplicate IDs with `keep=False`. Here they are handled more conservatively: one canonical row is kept and extra copies are quarantined.

In [ ]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception as exc:
    raise RuntimeError(
        "Run this notebook inside Snowflake, or replace this block with a configured "
        "Snowpark Session. No credentials should be written into the notebook."
    ) from exc

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

# Edit only these object names. Identifiers are validated before use.
RAW_TABLE = "CRIME_ETL_DB.RAW.STREET_CRIME"
WORK_SCHEMA = "CRIME_ETL_DB.DATA_QUALITY"
CLEAN_TABLE = f"{WORK_SCHEMA}.STREET_CRIME_CLEAN"
QUARANTINE_TABLE = f"{WORK_SCHEMA}.STREET_CRIME_QUARANTINE"
RUN_AUDIT_TABLE = f"{WORK_SCHEMA}.CLEANING_RUN_AUDIT"
FORCE_AUDIT_TABLE = f"{WORK_SCHEMA}.CLEANING_FORCE_AUDIT"
VALIDATION_TABLE = f"{WORK_SCHEMA}.VALIDATION_RESULTS"

IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*){0,2}$")
for name in [
    RAW_TABLE, WORK_SCHEMA, CLEAN_TABLE, QUARANTINE_TABLE,
    RUN_AUDIT_TABLE, FORCE_AUDIT_TABLE, VALIDATION_TABLE
]:
    if not IDENTIFIER.fullmatch(name):
        raise ValueError(f"Unsafe Snowflake object identifier: {name}")

print(f"Run ID: {RUN_ID}")
print(f"Source: {RAW_TABLE}")

## 1. Confirm the input contract

The source table must contain these columns (case-insensitive). `SOURCE_FILE_NAME` is recommended for lineage but is not required.

In [ ]:

REQUIRED_COLUMNS = {
    "CRIME ID", "MONTH", "REPORTED BY", "FALLS WITHIN", "LONGITUDE", "LATITUDE",
    "LOCATION", "LSOA CODE", "LSOA NAME", "CRIME TYPE", "LAST OUTCOME CATEGORY"
}

def show_column_name(row):
    values = row.as_dict()
    for key, value in values.items():
        if key.lower() in {"column_name", "name"}:
            return str(value)
    raise KeyError(f"Could not find column-name field in SHOW COLUMNS output: {values.keys()}")

source_columns = {
    show_column_name(r).upper()
    for r in session.sql(f"SHOW COLUMNS IN TABLE {RAW_TABLE}").collect()
}
missing_columns = REQUIRED_COLUMNS - source_columns
if missing_columns:
    raise ValueError(f"Source contract failed. Missing columns: {sorted(missing_columns)}")

HAS_SOURCE_FILE = "SOURCE_FILE_NAME" in source_columns
print("PASS - source contract satisfied")
print(f"SOURCE_FILE_NAME available: {HAS_SOURCE_FILE}")

## 2. Create audit structures


In [ ]:
session.sql(f"CREATE SCHEMA IF NOT EXISTS {WORK_SCHEMA}").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {RUN_AUDIT_TABLE} (
    RUN_ID STRING, STARTED_AT TIMESTAMP_TZ, COMPLETED_AT TIMESTAMP_TZ,
    SOURCE_TABLE STRING, CLEAN_TABLE STRING, QUARANTINE_TABLE STRING,
    RAW_TARGET_ROWS NUMBER, CLEAN_ROWS NUMBER, QUARANTINE_ROWS NUMBER,
    ROWS_REMOVED NUMBER, REMOVAL_PERCENT NUMBER(10,4), STATUS STRING
)
""").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {FORCE_AUDIT_TABLE} (
    RUN_ID STRING, FORCE_NAME STRING, RAW_ROWS NUMBER, CLEAN_ROWS NUMBER,
    QUARANTINE_ROWS NUMBER, DUPLICATE_ROWS NUMBER, NULL_CRIME_ID_ROWS NUMBER,
    NULL_MONTH_ROWS NUMBER, NULL_LSOA_ROWS NUMBER, NULL_COORDINATE_ROWS NUMBER
)
""").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDATION_TABLE} (
    RUN_ID STRING, CHECK_NAME STRING, SEVERITY STRING, OBSERVED_VALUE STRING,
    EXPECTED_VALUE STRING, PASSED BOOLEAN, CHECKED_AT TIMESTAMP_TZ
)
""").collect()
print("Audit structures ready")

## 3. Standardise and assess quality

Only the four in-scope forces are selected. The temporary stage exists only for this Snowflake session and does not modify the raw data.

In [ ]:
source_file_expression = (
    'NULLIF(TRIM("SOURCE_FILE_NAME"), \'\')'
    if HAS_SOURCE_FILE else "CAST(NULL AS STRING)"
)

session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE STG_POLICE_CRIME_STANDARDISED AS
WITH NORMALISED AS (
    SELECT
        NULLIF(TRIM("Crime ID"), '') AS CRIME_ID,
        TRY_TO_DATE(NULLIF(TRIM("Month"), '') || '-01') AS CRIME_MONTH,
        NULLIF(TRIM("Reported by"), '') AS REPORTED_BY_RAW,
        NULLIF(TRIM("Falls within"), '') AS FALLS_WITHIN_RAW,
        TRY_TO_DOUBLE("Longitude") AS LONGITUDE,
        TRY_TO_DOUBLE("Latitude") AS LATITUDE,
        NULLIF(TRIM("Location"), '') AS LOCATION,
        UPPER(NULLIF(TRIM("LSOA code"), '')) AS LSOA_CODE,
        NULLIF(TRIM("LSOA name"), '') AS LSOA_NAME,
        NULLIF(TRIM("Crime type"), '') AS CRIME_TYPE_RAW,
        NULLIF(TRIM("Last outcome category"), '') AS LAST_OUTCOME_CATEGORY,
        {source_file_expression} AS SOURCE_FILE_NAME,
        CURRENT_TIMESTAMP() AS CLEANED_AT,
        '{RUN_ID}' AS CLEANING_RUN_ID
    FROM {RAW_TABLE}
    WHERE LOWER(TRIM("Falls within")) IN (
        'metropolitan police', 'metropolitan police service',
        'west midlands police', 'south wales police', 'sussex police'
    )
), STANDARDISED AS (
    SELECT *,
        CASE
            WHEN LOWER(FALLS_WITHIN_RAW) IN
                ('metropolitan police', 'metropolitan police service')
                THEN 'Metropolitan Police Service'
            WHEN LOWER(FALLS_WITHIN_RAW) = 'west midlands police'
                THEN 'West Midlands Police'
            WHEN LOWER(FALLS_WITHIN_RAW) = 'south wales police'
                THEN 'South Wales Police'
            WHEN LOWER(FALLS_WITHIN_RAW) = 'sussex police'
                THEN 'Sussex Police'
        END AS FORCE_NAME,
        CASE LOWER(CRIME_TYPE_RAW)
            WHEN 'anti social behaviour' THEN 'Anti-social behaviour'
            WHEN 'antisocial behaviour' THEN 'Anti-social behaviour'
            WHEN 'anti-social behaviour' THEN 'Anti-social behaviour'
            WHEN 'bicycle theft' THEN 'Bicycle theft'
            WHEN 'burglary' THEN 'Burglary'
            WHEN 'criminal damage and arson' THEN 'Criminal damage and arson'
            WHEN 'drugs' THEN 'Drugs'
            WHEN 'other crime' THEN 'Other crime'
            WHEN 'other theft' THEN 'Other theft'
            WHEN 'possession of weapons' THEN 'Possession of weapons'
            WHEN 'public order' THEN 'Public order'
            WHEN 'robbery' THEN 'Robbery'
            WHEN 'shoplifting' THEN 'Shoplifting'
            WHEN 'theft from the person' THEN 'Theft from the person'
            WHEN 'vehicle crime' THEN 'Vehicle crime'
            WHEN 'violence & sexual offences' THEN 'Violence and sexual offences'
            WHEN 'violence and sexual offences' THEN 'Violence and sexual offences'
            ELSE CRIME_TYPE_RAW
        END AS CRIME_TYPE
    FROM NORMALISED
), ASSESSED AS (
    SELECT *,
        ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
            IFF(CRIME_ID IS NULL, 'MISSING_CRIME_ID', NULL),
            IFF(CRIME_MONTH IS NULL, 'INVALID_OR_MISSING_MONTH', NULL),
            IFF(FORCE_NAME IS NULL, 'UNKNOWN_FORCE', NULL),
            IFF(CRIME_TYPE IS NULL, 'MISSING_CRIME_TYPE', NULL),
            IFF(LSOA_CODE IS NULL, 'MISSING_LSOA_CODE', NULL),
            IFF(LATITUDE IS NULL OR LONGITUDE IS NULL, 'MISSING_COORDINATE', NULL),
            IFF(LATITUDE IS NOT NULL AND NOT (LATITUDE BETWEEN 49 AND 61),
                'LATITUDE_OUT_OF_UK_RANGE', NULL),
            IFF(LONGITUDE IS NOT NULL AND NOT (LONGITUDE BETWEEN -9 AND 3),
                'LONGITUDE_OUT_OF_UK_RANGE', NULL)
        ), '|') AS BASE_REJECTION_REASON,
        ROW_NUMBER() OVER (
            PARTITION BY CRIME_ID
            ORDER BY
                IFF(CRIME_MONTH IS NOT NULL, 1, 0)
                + IFF(LSOA_CODE IS NOT NULL, 1, 0)
                + IFF(LATITUDE IS NOT NULL, 1, 0)
                + IFF(LONGITUDE IS NOT NULL, 1, 0) DESC,
                SOURCE_FILE_NAME NULLS LAST, FORCE_NAME
        ) AS CRIME_ID_RANK
    FROM STANDARDISED
)
SELECT *,
    CASE
        WHEN BASE_REJECTION_REASON <> '' AND CRIME_ID_RANK > 1
            THEN BASE_REJECTION_REASON || '|DUPLICATE_CRIME_ID'
        WHEN BASE_REJECTION_REASON <> '' THEN BASE_REJECTION_REASON
        WHEN CRIME_ID_RANK > 1 THEN 'DUPLICATE_CRIME_ID'
        ELSE NULL
    END AS REJECTION_REASON
FROM ASSESSED
""").collect()

raw_target_rows = session.table("STG_POLICE_CRIME_STANDARDISED").count()
if raw_target_rows == 0:
    raise ValueError("No rows found for the four target forces; check source table and names.")
print(f"Rows assessed: {raw_target_rows:,}")

# Continue other Layers layout here